1 Setup libraries

In [ ]:
import json
import pandas as pd
from sklearn.ensemble import IsolationForest

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)


In [ ]:

# Define file paths (this is my path, change to your)
input_file = r"C:\Users\ADMIN\Documents\Algorithm_Inno\ML\data\data_processed.json"
output_file = r"C:\Users\ADMIN\Documents\Algorithm_Inno\ML\data\data_labeled.json"

2 Read data

In [ ]:
# Read data
df = pd.read_json(input_file)

if 'Timestamp' in df.columns:
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    df['Hour'] = df['Timestamp'].dt.hour
    df['DayOfWeek'] = df['Timestamp'].dt.dayofweek

df.head() 


3 Picking features 

In [ ]:
features = [
    'Transaction amount', 'Account balance', 'Salary (per month)',
    'Hour', 'DayOfWeek', 'Age', 'Is_Weekend', 'Is_Night',
    'Balance_to_Salary_Ratio', 'Tx_to_Balance_Ratio',
    'Transaction Detail', 'Geological', 'Device Use', 
    'Location', 'Working Status', 'Gender', 'Transaction Count'
]

selected_features = [col for col in features if col in df.columns]
X = df[selected_features]

4 Train model

In [ ]:
model = IsolationForest(n_estimators=100, contamination=0.15, random_state=42)
model.fit(X)

Number of Anomalies

In [ ]:
predictions = model.predict(X)
df['is_fraud'] = [1 if p == -1 else 0 for p in predictions]
df['anomaly_score'] = model.score_samples(X)

print(f"Found {df['is_fraud'].sum()} suspicious transactions.")


5 Saving result to json file (data_labeled)

In [ ]:

if 'DateTime' in df.columns:
    df['DateTime'] = df['DateTime'].astype(str)
elif 'Timestamp' in df.columns:
    df['Timestamp'] = df['Timestamp'].astype(str)
    
df.to_json(output_file, orient='records', indent=4, force_ascii=False)


6 Visualization


6.1 Anomaly Score Distribution (most important)

Plot a histogram of anomaly_score. You'll see a bimodal distribution — normal transactions cluster on the right (scores near 0), anomalies cluster on the left (very negative scores).

In [ ]:
import matplotlib.pyplot as plt
plt.hist(df['anomaly_score'], bins=50, color='steelblue')
plt.axvline(df[df['is_fraud']==1]['anomaly_score'].max(), color='red', linestyle='--', label='Fraud threshold')
plt.title('Anomaly Score Distribution')
plt.xlabel('Anomaly Score'); plt.legend(); plt.show()

6.2 Fraud vs Normal Count (Bar Chart)
    Simple but effective — shows how many transactions were flagged.

In [ ]:
df['is_fraud'].value_counts().rename({0:'Normal', 1:'Fraud'}).plot(kind='bar', color=['green','red'])
plt.title('Normal vs Fraud Count')


6.3 Transaction Amount vs Account Balance (Scatter Plot)
    Color points by is_fraud to see if anomalies cluster in specific regions (e.g., huge amounts with low balance).

In [ ]:
plt.scatter(df['Transaction amount'], df['Account balance'], c=df['is_fraud'], cmap='coolwarm', alpha=0.5)
plt.xlabel('Transaction Amount'); plt.ylabel('Account Balance')
plt.title('Transaction Amount vs Account Balance')


6.4 Anomaly Heatmap by Hour and Day of Week
    Shows when fraud happens — useful since code extracts Hour and DayOfWeek.

In [ ]:
import seaborn as sns
pivot = df[df['is_fraud']==1].groupby(['DayOfWeek','Hour']).size().unstack(fill_value=0)
sns.heatmap(pivot, cmap='Reds')
plt.title('Fraud Heatmap: Hour vs Day of Week')
